In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path("/content/drive/MyDrive/fee-defaulter-prediction")
(project_root / "data" / "processed").mkdir(parents= True, exist_ok=True)

print(project_root)

/content/drive/MyDrive/fee-defaulter-prediction


In [10]:
input_path = project_root / "data" / "processed" / "01_eda_output.csv"
df = pd.read_csv(input_path)

print(df.shape)
df.head()

(395, 23)


,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,traveltime,...,famsup,paid,higher,internet,famrel,absences,G1,G2,G3,default
0,F,18,U,GT3,A,4,4,at_home,teacher,2,...,no,no,yes,no,4,6,5,6,6,1
1,F,17,U,GT3,T,1,1,at_home,other,1,...,yes,no,yes,yes,5,4,5,5,6,1
2,F,15,U,LE3,T,1,1,at_home,other,1,...,no,yes,yes,yes,4,10,7,8,10,1
3,F,15,U,GT3,T,4,2,health,services,1,...,yes,yes,yes,yes,3,2,15,14,15,0
4,F,16,U,GT3,T,3,3,other,other,1,...,yes,yes,yes,no,4,4,6,10,10,0


In [15]:
df_fe = df.copy()

binary_maps ={
    "sex": {"F": 0, "M": 1},
    "address": {"R": 0, "U": 1},
    "famsize": {"LE3": 0, "GT3": 1},
    "Pstatus": {"A": 0, "T": 1},
    "schoolsup": {"no": 0, "yes": 1},
    "famsup": {"no": 0, "yes": 1},
    "paid": {"no": 0, "yes": 1},
    "higher": {"no": 0, "yes": 1},
    "internet": {"no": 0, "yes": 1},
}

for col, mapping in binary_maps.items():
  df_fe[col] = df_fe[col].map(mapping)

df_fe = pd.get_dummies(df_fe, columns=["Mjob", "Fjob"], drop_first=True)

df_fe["parent_edu_total"] = df_fe["Medu"] + df_fe["Fedu"]
df_fe["parent_edu_gap"] = (df_fe["Medu"] - df_fe["Fedu"]).abs()
df_fe["grade_avg_12"] = (df_fe["G1"] + df_fe["G2"]) / 2
df_fe["grade_delta_12"] = df_fe["G1"] - df_fe["G2"]
df_fe["support_total"] = df_fe["schoolsup"] + df_fe["famsup"]
df_fe["resource_support_total"] = df_fe["schoolsup"] + df_fe["famsup"] + df_fe["paid"]
df_fe["access_risk_score"] = df_fe["traveltime"] + (1 -df_fe["internet"])
df_fe["family_stability_score"] = df_fe['Pstatus'] + df_fe['famrel']

df_fe.head()

,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,failures,...,Fjob_services,Fjob_teacher,parent_edu_total,parent_edu_gap,grade_avg_12,grade_delta_12,support_total,resource_support_total,access_risk_score,family_stability_score
0,0,18,1,1,0,4,4,2,2,0,...,False,True,8,0,5.5,-1,1,1,3,4
1,0,17,1,1,1,1,1,1,2,0,...,False,False,2,0,5.0,0,1,1,1,6
2,0,15,1,0,1,1,1,1,2,3,...,False,False,2,0,7.5,-1,1,2,1,5
3,0,15,1,1,1,4,2,1,3,0,...,True,False,6,2,14.5,1,1,2,1,4
4,0,16,1,1,1,3,3,1,2,0,...,False,False,6,0,8.0,-4,1,2,2,5


In [16]:
target_col = "default"
leakage_cols = ["G3", "failures", "absences"]

df_full = df_fe.copy()
df_strict = df_fe.drop(columns=leakage_cols).copy()

print("Full dataset shape:", df_full.shape)
print("Strict dadtaset shape:", df_strict.shape)

Full dataset shape: (395, 37)
Strict dadtaset shape: (395, 34)


## Feature Engineering Output Summary

After encoding and feature creation, two modeling datasets were prepared:

- **Full dataset**: includes all available engineered predictors
- **Strict dataset**: excludes 'G3', 'failures', and 'absences' to reduce direct leakage from the proxy target construction.

The final stage were:
- Full stage: 395 rows × 37 columns
- Strict dataset: 395 × 34 columns

In [18]:
full_path = project_root / "data" / "processed" / "02_features_full.csv"
strict_path = project_root / "data" / "processed" / "02_features_strict.csv"

df_full.to_csv(full_path, index=False)
df_strict.to_csv(strict_path, index=False)